# Self-Supervised Multi-Head Attention: Sentiment & Intent Classification

## Strategy
This notebook adds a **self-supervised pretraining stage** before the supervised joint-head fine-tuning.

### Two-Stage Pipeline
```
Stage 1 – Self-Supervised Pretraining (MLM)
  Raw text  →  Mask 15% tokens  →  Predict masked tokens
  (no labels needed — the model learns language structure)

Stage 2 – Supervised Fine-Tuning (Joint Heads)
  Pretrained encoder  →  Sentiment head + Intent head
  (uses your labeled CSV)
```

### Why Self-Supervision Helps Here
- Your dataset is small (1000 rows). SSL lets the encoder learn domain vocabulary (Sinhala-Latin romanized text) before seeing any labels.
- MLM forces the model to understand context from both directions (same as how BERT was originally trained).
- The pretrained encoder weights are then transferred to the joint multi-head classifier.


## 0. Install Dependencies

In [2]:
!pip install transformers datasets accelerate -q

## 1. Imports & Configuration

In [3]:
import os, re, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForMaskedLM,
    AutoModel,
    DataCollatorForLanguageModeling,
    get_linear_schedule_with_warmup,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
warnings.filterwarnings('ignore')

# ── Configuration ──────────────────────────────────────────────────────────────
MODEL_NAME       = 'xlm-roberta-base'   # same backbone as the original notebook
MAX_LEN          = 128
MLM_PROB         = 0.15                 # fraction of tokens masked during SSL

# Stage 1 – MLM pretraining
MLM_EPOCHS       = 3
MLM_BATCH        = 16
MLM_LR           = 2e-5

# Stage 2 – Joint fine-tuning
FT_EPOCHS        = 5
FT_BATCH         = 16
FT_LR            = 2e-5
DROPOUT          = 0.3
NUM_ATTN_HEADS   = 4                    # heads in the task-specific attention layer

SSL_CKPT_DIR     = './ssl_pretrained'
FINAL_MODEL_DIR  = './ssl_joint_model'

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'PyTorch: {torch.__version__}')

Device: cuda
PyTorch: 2.10.0+cu128


## 2. Load & Preprocess Data

In [4]:
# ── Load CSV ───────────────────────────────────────────────────────────────────
# Update path for your environment
CSV_PATH = '/kaggle/input/datasets/thiwankachanditha/sold-data-sentiment-intent/sinhala_silver_labels.csv'
# CSV_PATH = 'sentiment_intent_predicted_1000.csv'  # local fallback
df = pd.read_csv(CSV_PATH)

print(df.shape)
df.head()

(5881, 8)


,Sentence,Sentiment,intent_label,combined_label,nli_conf,conf_tier,nli_rule_agree,sample_weight
0,Ammage Adarayta❤️Eka Dawasak Madi Neda❤️🙏❤️,Negative,inquiry,Negative_inquiry,0.3559,rule,True,1.0
1,We need IPL Champions leak data offers ..pleas...,Neutral,inquiry,Neutral_inquiry,0.7913,high,False,1.0
2,chandimal.. uuu thama mulu tem ekama kaaa gaha...,Positive,inquiry,Positive_inquiry,0.4922,rule,False,1.0
3,Lebsack 49k dammama eka dawasak wath be,Negative,inquiry,Negative_inquiry,0.8232,rule,True,1.0
4,eth anith kattiya sathiyak withara online inna...,Neutral,other,Neutral_other,0.6918,mid,False,0.5


In [6]:
df.rename(columns={'Sentence': 'text', 'intent': 'intent_label',
                   'Sentiment': 'sentiment_label'}, inplace=True)

print(df.shape)
df.head()

(5881, 8)


,text,sentiment_label,intent_label,combined_label,nli_conf,conf_tier,nli_rule_agree,sample_weight
0,Ammage Adarayta❤️Eka Dawasak Madi Neda❤️🙏❤️,Negative,inquiry,Negative_inquiry,0.3559,rule,True,1.0
1,We need IPL Champions leak data offers ..pleas...,Neutral,inquiry,Neutral_inquiry,0.7913,high,False,1.0
2,chandimal.. uuu thama mulu tem ekama kaaa gaha...,Positive,inquiry,Positive_inquiry,0.4922,rule,False,1.0
3,Lebsack 49k dammama eka dawasak wath be,Negative,inquiry,Negative_inquiry,0.8232,rule,True,1.0
4,eth anith kattiya sathiyak withara online inna...,Neutral,other,Neutral_other,0.6918,mid,False,0.5


In [7]:
# ── Text cleaning (same as original notebook) ──────────────────────────────────
def clean_text(text: str) -> str:
    text = str(text).lower().strip()
    text = re.sub(r'http\S+|www\S+', '', text)          # URLs
    text = re.sub(r'[^\w\s\u0D80-\u0DFF]', ' ', text)  # keep Sinhala unicode
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['text'] = df['text'].apply(clean_text)
df = df.dropna(subset=['text', 'sentiment_label', 'intent_label'])
df = df[df['text'].str.len() > 3].reset_index(drop=True)

# ── Encode labels ──────────────────────────────────────────────────────────────
sent_enc   = LabelEncoder()
intent_enc = LabelEncoder()
df['sent_id']   = sent_enc.fit_transform(df['sentiment_label'])
df['intent_id'] = intent_enc.fit_transform(df['intent_label'])

SENTIMENT_CLASSES = list(sent_enc.classes_)
INTENT_CLASSES    = list(intent_enc.classes_)
NUM_SENT          = len(SENTIMENT_CLASSES)
NUM_INTENT        = len(INTENT_CLASSES)

print('Sentiment classes :', SENTIMENT_CLASSES)
print('Intent classes    :', INTENT_CLASSES)

# ── Train / val split ──────────────────────────────────────────────────────────
train_df, val_df = train_test_split(df, test_size=0.15, random_state=42,
                                     stratify=df['sent_id'])
print(f'Train: {len(train_df)}  Val: {len(val_df)}')

Sentiment classes : ['Conflict', 'Negative', 'Neutral', 'Positive']
Intent classes    : ['complaint', 'feedback', 'greeting', 'inquiry', 'other', 'suggestion']
Train: 4986  Val: 881


## 3. Stage 1 — Self-Supervised Pretraining (MLM)

We treat **all raw text** (train + val) as an **unlabeled corpus** and train the encoder
to predict randomly masked tokens.  
This is identical to how BERT/XLM-RoBERTa were originally pretrained, applied to your
specific domain text (Sinhala romanized + mixed-script).

```
Input : "meka harima [MASK] mata refund ekk one"
Target:                wadiyai
```

After this stage the encoder understands the vocabulary and patterns of your corpus
**without seeing any sentiment/intent labels**.

In [8]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [9]:
class MLMDataset(Dataset):
    """
    Returns tokenized encodings WITHOUT labels.
    The DataCollatorForLanguageModeling will dynamically mask tokens
    at training time (fresh random mask each epoch — better generalisation).
    """
    def __init__(self, texts, tokenizer, max_len):
        self.encodings = tokenizer(
            list(texts),
            max_length=max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )

    def __len__(self):
        return self.encodings['input_ids'].shape[0]

    def __getitem__(self, idx):
        # Return only input_ids and attention_mask; collator adds labels
        return {
            'input_ids':      self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
        }

# Use ALL text for SSL (labels not needed)
all_texts   = df['text'].tolist()
mlm_dataset = MLMDataset(all_texts, tokenizer, MAX_LEN)

# Dynamic masking collator — randomly masks MLM_PROB fraction each batch
mlm_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=MLM_PROB,
    pad_to_multiple_of=None,
    return_tensors='pt',
)

mlm_loader = DataLoader(
    mlm_dataset,
    batch_size=MLM_BATCH,
    shuffle=True,
    collate_fn=mlm_collator,
)
print(f'MLM batches per epoch: {len(mlm_loader)}')

MLM batches per epoch: 367


In [10]:
# ── Load XLM-RoBERTa with MLM head ────────────────────────────────────────────
mlm_model = AutoModelForMaskedLM.from_pretrained(MODEL_NAME).to(DEVICE)
mlm_optimizer = torch.optim.AdamW(mlm_model.parameters(), lr=MLM_LR)
total_steps = MLM_EPOCHS * len(mlm_loader)
mlm_scheduler = get_linear_schedule_with_warmup(
    mlm_optimizer,
    num_warmup_steps=max(1, total_steps // 10),
    num_training_steps=total_steps,
)

print(f'Starting MLM pretraining for {MLM_EPOCHS} epoch(s)...')
print(f'Total texts in corpus: {len(all_texts)}')

mlm_model.train()
for epoch in range(MLM_EPOCHS):
    total_loss = 0.0
    for step, batch in enumerate(mlm_loader):
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels         = batch['labels'].to(DEVICE)  # -100 for non-masked tokens

        mlm_optimizer.zero_grad()
        outputs = mlm_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )
        loss = outputs.loss
        loss.backward()
        nn.utils.clip_grad_norm_(mlm_model.parameters(), 1.0)
        mlm_optimizer.step()
        mlm_scheduler.step()
        total_loss += loss.item()

        if (step + 1) % 10 == 0:
            print(f'  Epoch {epoch+1}/{MLM_EPOCHS}  Step {step+1}/{len(mlm_loader)}  '
                  f'Loss: {loss.item():.4f}')

    avg = total_loss / len(mlm_loader)
    print(f'Epoch {epoch+1} avg MLM loss: {avg:.4f}\n')

print('MLM pretraining complete!')

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

XLMRobertaForMaskedLM LOAD REPORT from: xlm-roberta-base
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Starting MLM pretraining for 3 epoch(s)...
Total texts in corpus: 5867
  Epoch 1/3  Step 10/367  Loss: 6.2069
  Epoch 1/3  Step 20/367  Loss: 5.2380
  Epoch 1/3  Step 30/367  Loss: 5.2409
  Epoch 1/3  Step 40/367  Loss: 4.7649
  Epoch 1/3  Step 50/367  Loss: 5.5045
  Epoch 1/3  Step 60/367  Loss: 4.7184
  Epoch 1/3  Step 70/367  Loss: 4.6660
  Epoch 1/3  Step 80/367  Loss: 4.6074
  Epoch 1/3  Step 90/367  Loss: 4.6186
  Epoch 1/3  Step 100/367  Loss: 3.9598
  Epoch 1/3  Step 110/367  Loss: 4.7939
  Epoch 1/3  Step 120/367  Loss: 3.2424
  Epoch 1/3  Step 130/367  Loss: 4.8302
  Epoch 1/3  Step 140/367  Loss: 3.8732
  Epoch 1/3  Step 150/367  Loss: 3.8664
  Epoch 1/3  Step 160/367  Loss: 5.7344
  Epoch 1/3  Step 170/367  Loss: 4.1431
  Epoch 1/3  Step 180/367  Loss: 3.1410
  Epoch 1/3  Step 190/367  Loss: 3.6769
  Epoch 1/3  Step 200/367  Loss: 4.2049
  Epoch 1/3  Step 210/367  Loss: 4.9004
  Epoch 1/3  Step 220/367  Loss: 4.8878
  Epoch 1/3  Step 230/367  Loss: 3.8160
  Epoch 1/3  Step 

OutOfMemoryError: CUDA out of memory. Tried to allocate 1.91 GiB. GPU 0 has a total capacity of 14.56 GiB of which 659.81 MiB is free. Including non-PyTorch memory, this process has 13.92 GiB memory in use. Of the allocated memory 10.06 GiB is allocated by PyTorch, and 3.72 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
# ── Save the SSL-pretrained encoder (not the MLM head) ────────────────────────
os.makedirs(SSL_CKPT_DIR, exist_ok=True)
mlm_model.base_model.save_pretrained(SSL_CKPT_DIR)   # saves only the transformer encoder
tokenizer.save_pretrained(SSL_CKPT_DIR)
print(f'SSL encoder saved to: {SSL_CKPT_DIR}')
for f in os.listdir(SSL_CKPT_DIR):
    mb = os.path.getsize(f'{SSL_CKPT_DIR}/{f}') / 1e6
    print(f'  {f}  ({mb:.1f} MB)')

## 4. Stage 2 — Supervised Fine-Tuning with Multi-Head Attention

We load the **SSL-pretrained encoder** (from Stage 1) and attach two task-specific heads:
- A **multi-head self-attention pooling layer** over the token sequence
- A **sentiment classification head**
- An **intent classification head**

The multi-head attention pooling replaces simple [CLS] pooling — it learns to attend
to the most informative tokens for each task simultaneously.

```
Encoder (SSL-pretrained)
       ↓  [seq_len × 768]
Multi-Head Attention Pooling  ← learns task-relevant token importance
       ↓  [768]
  ┌────┴────┐
Sentiment  Intent
  Head      Head
```

In [ ]:
class SentimentIntentDataset(Dataset):
    def __init__(self, df, tokenizer, max_len):
        self.texts      = df['text'].tolist()
        self.sent_ids   = df['sent_id'].tolist()
        self.intent_ids = df['intent_id'].tolist()
        self.tokenizer  = tokenizer
        self.max_len    = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'sent_label':     torch.tensor(self.sent_ids[idx],   dtype=torch.long),
            'intent_label':   torch.tensor(self.intent_ids[idx], dtype=torch.long),
        }

train_dataset = SentimentIntentDataset(train_df, tokenizer, MAX_LEN)
val_dataset   = SentimentIntentDataset(val_df,   tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=FT_BATCH, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=FT_BATCH, shuffle=False)
print(f'Train batches: {len(train_loader)}  Val batches: {len(val_loader)}')

In [ ]:
class MultiHeadAttentionPooling(nn.Module):
    """
    Replaces [CLS] pooling with a learnable multi-head attention over all tokens.
    A query vector learns which tokens are most relevant for the downstream tasks.

    Shape:
      Input  : (B, L, H)  — batch × seq_len × hidden_size
      Output : (B, H)     — batch × hidden_size
    """
    def __init__(self, hidden_size: int, num_heads: int, dropout: float):
        super().__init__()
        self.attn = nn.MultiheadAttention(
            embed_dim=hidden_size,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True,
        )
        # Learnable query: one token that "asks" the sequence for task-relevant info
        self.query = nn.Parameter(torch.randn(1, 1, hidden_size))
        self.norm  = nn.LayerNorm(hidden_size)

    def forward(self, hidden_states, attention_mask=None):
        # hidden_states : (B, L, H)
        B = hidden_states.size(0)
        q = self.query.expand(B, -1, -1)            # (B, 1, H)

        # Convert HuggingFace attention mask (1=keep, 0=pad) to PyTorch key_padding_mask
        key_padding_mask = None
        if attention_mask is not None:
            key_padding_mask = (attention_mask == 0)  # True = ignore

        attn_out, _ = self.attn(
            query=q,
            key=hidden_states,
            value=hidden_states,
            key_padding_mask=key_padding_mask,
        )                                            # (B, 1, H)
        pooled = attn_out.squeeze(1)                 # (B, H)
        return self.norm(pooled)


class SSLJointModel(nn.Module):
    """
    SSL-pretrained encoder + Multi-Head Attention Pooling
    + shared dense + sentiment head + intent head.
    """
    def __init__(
        self,
        encoder_path: str,
        num_sentiment: int,
        num_intent: int,
        num_heads: int  = NUM_ATTN_HEADS,
        dropout: float  = DROPOUT,
    ):
        super().__init__()
        self.encoder   = AutoModel.from_pretrained(encoder_path)
        hidden         = self.encoder.config.hidden_size   # 768 for base models

        self.pool      = MultiHeadAttentionPooling(hidden, num_heads, dropout)
        self.shared_fc = nn.Sequential(
            nn.Linear(hidden, hidden // 2),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.sent_head   = nn.Linear(hidden // 2, num_sentiment)
        self.intent_head = nn.Linear(hidden // 2, num_intent)

    def forward(self, input_ids, attention_mask):
        enc_out  = self.encoder(input_ids=input_ids,
                                attention_mask=attention_mask)
        seq_out  = enc_out.last_hidden_state        # (B, L, H)
        pooled   = self.pool(seq_out, attention_mask)  # (B, H)
        shared   = self.shared_fc(pooled)           # (B, H/2)
        sent_logits   = self.sent_head(shared)      # (B, num_sentiment)
        intent_logits = self.intent_head(shared)    # (B, num_intent)
        return sent_logits, intent_logits


# ── Instantiate model from SSL checkpoint ─────────────────────────────────────
model = SSLJointModel(
    encoder_path=SSL_CKPT_DIR,
    num_sentiment=NUM_SENT,
    num_intent=NUM_INTENT,
).to(DEVICE)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params   : {total_params:,}')
print(f'Trainable params: {trainable_params:,}')

In [ ]:
# ── Optimizer & scheduler ──────────────────────────────────────────────────────
# Use different learning rates: lower for pretrained encoder, higher for new heads
optimizer = torch.optim.AdamW([
    {'params': model.encoder.parameters(),   'lr': FT_LR * 0.1},
    {'params': model.pool.parameters(),      'lr': FT_LR},
    {'params': model.shared_fc.parameters(), 'lr': FT_LR},
    {'params': model.sent_head.parameters(), 'lr': FT_LR},
    {'params': model.intent_head.parameters(),'lr': FT_LR},
], weight_decay=0.01)

total_steps = FT_EPOCHS * len(train_loader)
scheduler   = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=max(1, total_steps // 10),
    num_training_steps=total_steps,
)
criterion = nn.CrossEntropyLoss()

# ── Training loop ──────────────────────────────────────────────────────────────
def evaluate(model, loader):
    model.eval()
    all_sent_true, all_sent_pred   = [], []
    all_intent_true, all_intent_pred = [], []
    total_loss = 0.0
    with torch.no_grad():
        for batch in loader:
            ids  = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            sl   = batch['sent_label'].to(DEVICE)
            il   = batch['intent_label'].to(DEVICE)
            slog, ilog = model(ids, mask)
            loss = criterion(slog, sl) + criterion(ilog, il)
            total_loss += loss.item()
            all_sent_true.extend(sl.cpu().tolist())
            all_sent_pred.extend(slog.argmax(-1).cpu().tolist())
            all_intent_true.extend(il.cpu().tolist())
            all_intent_pred.extend(ilog.argmax(-1).cpu().tolist())
    sent_acc   = np.mean(np.array(all_sent_true)   == np.array(all_sent_pred))
    intent_acc = np.mean(np.array(all_intent_true) == np.array(all_intent_pred))
    return total_loss / len(loader), sent_acc, intent_acc


print(f'Fine-tuning for {FT_EPOCHS} epoch(s) on {len(train_df)} samples...\n')
best_val_loss = float('inf')

for epoch in range(FT_EPOCHS):
    model.train()
    train_loss = 0.0
    for step, batch in enumerate(train_loader):
        ids  = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        sl   = batch['sent_label'].to(DEVICE)
        il   = batch['intent_label'].to(DEVICE)

        optimizer.zero_grad()
        slog, ilog = model(ids, mask)
        loss = criterion(slog, sl) + criterion(ilog, il)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        train_loss += loss.item()

    avg_train = train_loss / len(train_loader)
    val_loss, sent_acc, intent_acc = evaluate(model, val_loader)

    print(f'Epoch {epoch+1}/{FT_EPOCHS}  '
          f'Train Loss: {avg_train:.4f}  '
          f'Val Loss: {val_loss:.4f}  '
          f'Sent Acc: {sent_acc:.3f}  '
          f'Intent Acc: {intent_acc:.3f}')

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), './best_ssl_joint.pt')
        print('  ✓ Best model saved')

print('\nFine-tuning complete!')

## 5. Evaluation

In [ ]:
# Load best checkpoint
model.load_state_dict(torch.load('./best_ssl_joint.pt', map_location=DEVICE))
model.eval()

all_sent_true, all_sent_pred     = [], []
all_intent_true, all_intent_pred = [], []

with torch.no_grad():
    for batch in val_loader:
        ids  = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        slog, ilog = model(ids, mask)
        all_sent_true.extend(batch['sent_label'].tolist())
        all_sent_pred.extend(slog.argmax(-1).cpu().tolist())
        all_intent_true.extend(batch['intent_label'].tolist())
        all_intent_pred.extend(ilog.argmax(-1).cpu().tolist())

print('='*55)
print('SENTIMENT CLASSIFICATION REPORT')
print('='*55)
print(classification_report(
    all_sent_true, all_sent_pred,
    target_names=SENTIMENT_CLASSES, digits=4
))

print('='*55)
print('INTENT CLASSIFICATION REPORT')
print('='*55)
print(classification_report(
    all_intent_true, all_intent_pred,labels=list(range(len(INTENT_CLASSES))),
    target_names=INTENT_CLASSES, digits=4
))

## 6. Save Final Model

In [ ]:
import json

os.makedirs(FINAL_MODEL_DIR, exist_ok=True)
model.encoder.save_pretrained(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)
torch.save(model.state_dict(), f'{FINAL_MODEL_DIR}/ssl_joint_heads.pt')

# Save label mappings
meta = {
    'sentiment_classes': SENTIMENT_CLASSES,
    'intent_classes':    INTENT_CLASSES,
    'max_len':           MAX_LEN,
    'num_attn_heads':    NUM_ATTN_HEADS,
    'ssl_pretrained':    True,
    'mlm_epochs':        MLM_EPOCHS,
    'ft_epochs':         FT_EPOCHS,
}
with open(f'{FINAL_MODEL_DIR}/meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

print(f'Model saved to: {FINAL_MODEL_DIR}')
for fn in os.listdir(FINAL_MODEL_DIR):
    mb = os.path.getsize(f'{FINAL_MODEL_DIR}/{fn}') / 1e6
    print(f'  {fn}  ({mb:.1f} MB)')

## 7. Inference on New Text

In [ ]:
def predict(text: str) -> dict:
    model.eval()
    text = clean_text(str(text))
    enc  = tokenizer(
        text,
        max_length=MAX_LEN,
        padding='max_length',
        truncation=True,
        return_tensors='pt',
    )
    with torch.no_grad():
        slog, ilog = model(
            enc['input_ids'].to(DEVICE),
            enc['attention_mask'].to(DEVICE),
        )
    sp = torch.softmax(slog, -1).cpu().numpy()[0]
    ip = torch.softmax(ilog, -1).cpu().numpy()[0]
    return {
        'text':                 text,
        'sentiment':            SENTIMENT_CLASSES[sp.argmax()],
        'sentiment_confidence': round(float(sp.max()), 4),
        'intent':               INTENT_CLASSES[ip.argmax()],
        'intent_confidence':    round(float(ip.max()), 4),
        'sentiment_probs': {c: round(float(p), 4) for c, p in zip(SENTIMENT_CLASSES, sp)},
        'intent_probs':    {c: round(float(p), 4) for c, p in zip(INTENT_CLASSES,   ip)},
    }

def pretty_print(r: dict):
    print('\n' + '-'*55)
    print(f"  Text      : {r['text']}")
    print(f"  Sentiment : {r['sentiment']:<12} (conf: {r['sentiment_confidence']:.1%})")
    print(f"  Intent    : {r['intent']:<12} (conf: {r['intent_confidence']:.1%})")
    print('  Sentiment probs:')
    for c, p in r['sentiment_probs'].items():
        print(f'    {c:<14} {p:.4f}')
    print('  Intent probs:')
    for c, p in r['intent_probs'].items():
        print(f'    {c:<14} {p:.4f}')

samples = [
    'meka harima wadiyai mata refund ekk one',
    'oyage service eka best godak stuthi',
    'delivery date eka kawadad',
    'me product eka use karanne kohomad',
    'proud to be a sri lankan',
]
for text in samples:
    pretty_print(predict(text))

## 8. SSL vs Supervised-Only: What Changed?

| Aspect | Original (Supervised Only) | This Notebook (SSL + Fine-tuning) |
|---|---|---|
| **Stage 1** | ❌ None | ✅ MLM on raw text (no labels) |
| **Pooling** | [CLS] token | Multi-head attention pooling |
| **Label efficiency** | Needs all 1000 labeled rows | Stage 1 uses 1000 rows without labels |
| **Masking** | None | 15% dynamic token masking |
| **LR schedule** | Single LR | Differential LR (encoder 10× lower) |
| **Expected benefit** | Baseline | Better on domain vocab + low-data regime |

### Tips for Further Improvement
- **More unlabeled text**: Collect more Sinhala social-media text for Stage 1 — SSL scales with data volume.
- **Contrastive SSL (SimCSE)**: Instead of MLM, push semantically similar sentences closer in embedding space.
- **Longer MLM pretraining**: More epochs on domain text = stronger encoder before fine-tuning.
- **Task-specific attention heads**: Add separate attention pooling per task (sentiment vs intent) instead of a shared pooler.